# Inference-time interval slice diagnostics

Issue #20 follow-up. The 2025 outcomes have already been inspected in an earlier study, so this notebook does not use them to choose or validate a new method. Fit the fixed point model on 2020-2023, calibrate on January-June 2024, and compare on July-December 2024. All comparisons here are exploratory. The true-price bands below are diagnostics only; candidate grouping uses only predicted price or ZIP, both available at inference time. No interval is approved for serving.

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from homelens.data.inventory import _file_identity
from homelens.modeling.county_comparison import _fit, _predict

data_root = Path(os.environ.get('HOMELENS_DATA_ROOT', 'data/processed'))
cohort_path = data_root / 'modeling_cohort_county_verified.csv'
audit_path = data_root / 'modeling_cohort_county_verified_audit.json'
audit = json.loads(audit_path.read_text(encoding='utf-8'))
assert audit['rules']['county_boundary_validated'] is True
assert audit['split_boundaries'] == {'validation_start': '2024-01-01', 'test_start': '2025-01-01'}
cohort = pd.read_csv(cohort_path, dtype={'zip': 'string', 'property_type': 'string', 'split': 'string'})
assert len(cohort) == audit['prepared_rows']
assert cohort['split'].value_counts().to_dict() == audit['split_counts']
train = cohort.loc[cohort['split'].eq('train')].copy()
validation = cohort.loc[cohort['split'].eq('validation')].copy()
del cohort
assert pd.to_datetime(train['sale_date']).max() < pd.Timestamp('2024-01-01')
assert pd.to_datetime(validation['sale_date']).between('2024-01-01', '2024-12-31').all()
model, categories = _fit(train)
train_prediction = _predict(model, categories, train)
prediction = _predict(model, categories, validation)
actual = validation['price_usd'].to_numpy(dtype=float)
calibration = (pd.to_datetime(validation['sale_date']) < pd.Timestamp('2024-07-01')).to_numpy()
assessment = ~calibration
assert calibration.sum() >= 1000 and assessment.sum() >= 1000
assert np.isfinite(prediction).all() and (prediction > 0).all()
train_price_p50, train_price_p90 = train['price_usd'].quantile([0.5, 0.9]).to_numpy()
prediction_edges = np.quantile(train_prediction, [0.5, 0.9])
predicted_band = np.digitize(prediction, prediction_edges, right=True)
zip_code = validation['zip'].astype(str).to_numpy()
print({'train_rows': len(train), 'calibration_rows': int(calibration.sum()), 'assessment_rows': int(assessment.sum()), 'cohort_sha256': _file_identity(cohort_path)['sha256'], 'training_prediction_edges_usd': prediction_edges.round(2).tolist()})

{'train_rows': 13102, 'calibration_rows': 1514, 'assessment_rows': 1414, 'cohort_sha256': 'ecf3e0506958cecf7236996b995acd873e550c98ce358532a1d05364b71d320d', 'training_prediction_edges_usd': [350726.34, 562663.69]}


Use the previously fixed 9% lower / 1% upper tail allocation for each candidate. A local group needs a declared minimum calibration count; smaller groups use global scales. The 1% upper tail is inherently fragile for small groups, even when a support threshold passes.

In [2]:
def conformal_quantile(values, coverage):
    ordered = np.sort(np.asarray(values, dtype=float))
    assert len(ordered) and np.isfinite(ordered).all()
    rank = min(len(ordered), int(np.ceil((len(ordered) + 1) * coverage)))
    return float(ordered[rank - 1])


def scales(mask):
    error = actual[mask] - prediction[mask]
    return (conformal_quantile(np.maximum(-error, 0) / prediction[mask], 0.91), conformal_quantile(np.maximum(error, 0) / prediction[mask], 0.99))


global_scales = scales(calibration)

def candidate(groups=None, min_rows=0):
    lower_scale = np.full(len(prediction), global_scales[0])
    upper_scale = np.full(len(prediction), global_scales[1])
    if groups is not None:
        for group in np.unique(groups):
            members = groups == group
            cal_members = calibration & members
            if cal_members.sum() < min_rows:
                continue
            local_lower, local_upper = scales(cal_members)
            lower_scale[members] = local_lower
            upper_scale[members] = local_upper
    return np.maximum(0, prediction * (1 - lower_scale)), prediction * (1 + upper_scale)


intervals = {'global': candidate(), 'predicted_band_100': candidate(predicted_band, 100), 'zip_100': candidate(zip_code, 100), 'zip_60_exploratory': candidate(zip_code, 60)}
zip_support = pd.DataFrame([{'zip': code, 'calibration_rows': int((calibration & (zip_code == code)).sum()), 'assessment_rows': int((assessment & (zip_code == code)).sum()), 'uses_local_100': int((calibration & (zip_code == code)).sum()) >= 100, 'uses_local_60': int((calibration & (zip_code == code)).sum()) >= 60} for code in np.unique(zip_code)])
print('Global scales (lower, upper):', tuple(round(x, 4) for x in global_scales))
print('Predicted-band calibration rows:', {int(band): int((calibration & (predicted_band == band)).sum()) for band in range(3)})
print(zip_support.to_string(index=False))

Global scales (lower, upper): (0.1192, 0.7961)
Predicted-band calibration rows: {0: 696, 1: 644, 2: 174}
  zip  calibration_rows  assessment_rows  uses_local_100  uses_local_60
27503                17               12           False          False
27701                73               71           False           True
27703               560              536            True           True
27704               229              191            True           True
27705               199              173            True           True
27707               174              192            True           True
27712               155              130            True           True
27713               107              109            True           True


In [3]:
slices = {'overall': np.ones(len(actual), dtype=bool), 'actual_at_or_below_train_p50': actual <= train_price_p50, 'actual_above_train_p90': actual > train_price_p90, 'predicted_low': predicted_band == 0, 'predicted_high': predicted_band == 2}
for kind in np.unique(validation['property_type'].astype(str)):
    slices[f'type_{kind}'] = validation['property_type'].astype(str).to_numpy() == kind
for code in np.unique(zip_code):
    slices[f'zip_{code}'] = zip_code == code
rows = []
for method, (lower, upper) in intervals.items():
    for name, members in slices.items():
        selected = assessment & members
        count = int(selected.sum())
        row = {'method': method, 'slice': name, 'rows': count, 'small_slice': count < 30}
        if count >= 30:
            row.update(coverage=round(float(((actual[selected] >= lower[selected]) & (actual[selected] <= upper[selected])).mean()), 4), below_lower=round(float((actual[selected] < lower[selected]).mean()), 4), above_upper=round(float((actual[selected] > upper[selected]).mean()), 4), median_width_usd=round(float(np.median(upper[selected] - lower[selected])), 2))
        rows.append(row)
metrics = pd.DataFrame(rows)
focus = ['overall', 'actual_at_or_below_train_p50', 'actual_above_train_p90', 'predicted_low', 'predicted_high', 'zip_27701', 'zip_27707']
print(metrics.loc[metrics['slice'].isin(focus), ['method', 'slice', 'rows', 'coverage', 'below_lower', 'above_upper', 'median_width_usd']].to_string(index=False))
print('All ZIP coverage (small slices suppressed):')
print(metrics.loc[metrics['slice'].str.startswith('zip_')].pivot(index='slice', columns='method', values='coverage').to_string())

            method                        slice  rows  coverage  below_lower  above_upper  median_width_usd
            global                      overall  1414    0.8939       0.0976       0.0085         340288.67
            global actual_at_or_below_train_p50   452    0.8274       0.1726       0.0000         251987.97
            global       actual_above_train_p90   219    0.9041       0.0457       0.0502         563635.73
            global                predicted_low   616    0.9156       0.0812       0.0032         266610.06
            global               predicted_high   183    0.7760       0.1913       0.0328         597155.05
            global                    zip_27701    71    0.7183       0.2676       0.0141         363520.28
            global                    zip_27707   192    0.7760       0.1823       0.0417         444842.07
predicted_band_100                      overall  1414    0.8953       0.0976       0.0071         314488.36
predicted_band_100 actual_at

In [4]:
zip_support['upper_99pct_rank_from_max'] = zip_support['calibration_rows'].map(lambda n: n - min(n, int(np.ceil((n + 1) * 0.99))))
print('The upper-tail quantile uses the maximum when rank_from_max is zero:')
print(zip_support[['zip', 'calibration_rows', 'upper_99pct_rank_from_max']].to_string(index=False))
print('Property-type coverage and width:')
print(metrics.loc[metrics['slice'].str.startswith('type_'), ['method', 'slice', 'rows', 'coverage', 'median_width_usd']].to_string(index=False))
print('ZIP 27701/27707 width tradeoff:')
print(metrics.loc[metrics['slice'].isin(['zip_27701', 'zip_27707']), ['method', 'slice', 'rows', 'coverage', 'median_width_usd']].to_string(index=False))

The upper-tail quantile uses the maximum when rank_from_max is zero:
  zip  calibration_rows  upper_99pct_rank_from_max
27503                17                          0
27701                73                          0
27703               560                          4
27704               229                          1
27705               199                          1
27707               174                          0
27712               155                          0
27713               107                          0
Property-type coverage and width:
            method                          slice  rows  coverage  median_width_usd
            global               type_Condo/Co-op    49    0.9796         167378.56
            global type_Single Family Residential  1031    0.8739         368418.56
            global                 type_Townhouse   334    0.9431         302371.13
predicted_band_100               type_Condo/Co-op    49    0.9796         150800.96
predicted_band_100

## Decision

All candidates are research-only. The global interval covers 89.4% overall in July-December 2024, but only 82.7% of sales at or below the training-price median and 71.8% in ZIP 27701. Predicted-band calibration raises high-price coverage to 95.0%, but its high predicted band has a $1.26 million median interval width and low-price coverage falls to 80.1%. ZIP calibration with 100-row support cannot treat 27701 (73 calibration rows). Lowering support to 60 raises that ZIP's observed coverage to 91.5%, but its 99th-percentile upper scale is the single largest calibration score, making the result fragile; ZIP 27707 still covers only 84.9% with a $936,259 median width. The low-price band remains below 83% under every candidate.

Do not select an inference interval from these diagnostics. The reusable report should preserve the group support, fallbacks, coverage, and width rather than exporting scales or changing the API. A future model or calibration method needs a truly fresh later-period cohort to support a final claim, because 2025 outcomes were already inspected in the prior study.